# Milestone 1 — Colab CPU smoke test
Bản triển khai đang phát triển, chưa đạt toàn bộ đặc tả. Không chạy 5.000 ID. Chỉ chạy sau khi đã đọc README và các giới hạn. Mount Drive bằng giao diện Colab trước; notebook không tự đăng nhập/mount. Không Run all nếu chưa kiểm tra root.

In [ ]:
from pathlib import Path
import os, subprocess, sys
selected = Path(input('Thư mục dự án đã chọn trên Google Drive: ')).expanduser()
assert selected.is_dir(), 'Chọn thư mục Drive đã mount và tồn tại'
assert input('Xác nhận thư mục này nằm trên Drive (yes): ') == 'yes'
os.chdir(selected)
print('Project root selected')

In [ ]:
repo = Path('code')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/NewB-S2deeC0de/arXiv.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'src/requirements.txt')],check=True)
sys.path.insert(0,str(repo/'src'))

In [ ]:
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
config = load_config(Path('code/config/crawler.yaml'))
print('authentication=' + ('key' if load_semantic_scholar_key() else 'anonymous'))

## Smoke: tối đa 10 ID
Cell này gọi mạng thật khi bạn chủ động chạy. Dữ liệu nằm trong runtime/ tương đối với Drive root đã chọn.

In [ ]:
summary = await run_pipeline(config, {'arxiv','s2'}, resume=True, limit=10)
print(summary)

## Status và resume
Ngắt cell chạy bằng nút Stop, sau đó chạy lại cell smoke. Chỉ đặt takeover_stale=True nếu runtime trước đã dừng hoàn toàn.

In [ ]:
from crawler.progress import ProgressStore
from collections import Counter
store = ProgressStore(config.paths['persistent_root'])
print(Counter((r['stage'],r['status']) for r in store.rows.values()))

## Khi nhận API key
Thêm SEMANTIC_SCHOLAR_API_KEY trong Colab Secrets, bật quyền notebook. Chạy lại cell kiểm tra authentication, rồi cell smoke. Các stage failed chỉ được thử lại khi chủ động chạy cell sau; không reset dữ liệu thành công.

In [ ]:
# Chỉ chạy khi muốn mở chu kỳ retry mới cho failed stages.
# store.reset('failed')
# summary = await run_pipeline(config, {'arxiv','s2'}, resume=True, limit=10)

In [ ]:
from crawler.export import export_snapshot
import uuid
manifest = export_snapshot(config, 'export-' + uuid.uuid4().hex)
print(manifest)
